In [2]:
# Basic Import
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Modelling (Classification)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.svm import SVC

# Boosting Models
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

# Model Selection
from sklearn.model_selection import train_test_split, RandomizedSearchCV

# Metrics (Classification)
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve
)

import warnings
warnings.filterwarnings("ignore")

In [3]:
df=pd.read_csv('data/raw.csv')

In [4]:
df.head()

,Unnamed: 0,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [5]:
df['Churn'] = df['Churn'].map({'No': 0, 'Yes': 1})

In [6]:
cat_cols = df.select_dtypes(include='object').columns

for col in cat_cols:
    print(f"Categories in '{col}' variable:", df[col].unique())

Categories in 'customerID' variable: <StringArray>
['7590-VHVEG', '5575-GNVDE', '3668-QPYBK', '7795-CFOCW', '9237-HQITU',
 '9305-CDSKC', '1452-KIOVK', '6713-OKOMC', '7892-POOKP', '6388-TABGU',
 ...
 '9767-FFLEM', '0639-TSIQW', '8456-QDAVC', '7750-EYXWZ', '2569-WGERO',
 '6840-RESVB', '2234-XADUH', '4801-JZAZL', '8361-LTMKD', '3186-AJIEK']
Length: 7032, dtype: str
Categories in 'gender' variable: <StringArray>
['Female', 'Male']
Length: 2, dtype: str
Categories in 'Partner' variable: <StringArray>
['Yes', 'No']
Length: 2, dtype: str
Categories in 'Dependents' variable: <StringArray>
['No', 'Yes']
Length: 2, dtype: str
Categories in 'PhoneService' variable: <StringArray>
['No', 'Yes']
Length: 2, dtype: str
Categories in 'MultipleLines' variable: <StringArray>
['No phone service', 'No', 'Yes']
Length: 3, dtype: str
Categories in 'InternetService' variable: <StringArray>
['DSL', 'Fiber optic', 'No']
Length: 3, dtype: str
Categories in 'OnlineSecurity' variable: <StringArray>
['No', 'Yes', '

In [7]:
y=df['Churn']
X=df.drop(columns=['Churn'])

In [9]:
num_features=X.select_dtypes(exclude='object').columns
cat_features=X.select_dtypes(include='object').columns
cat_features

Index(['customerID', 'gender', 'Partner', 'Dependents', 'PhoneService',
       'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
       'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
       'Contract', 'PaperlessBilling', 'PaymentMethod'],
      dtype='str')

In [23]:
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.compose import ColumnTransformer
numeric_transformer=StandardScaler()
categoric_transformer=OneHotEncoder()
preprocessor=ColumnTransformer(
    [
        ("onehotencoder",categoric_transformer,cat_features),
        ("standardscaler",numeric_transformer,num_features)
    ]
)

In [24]:
X=preprocessor.fit_transform(X)

In [25]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [26]:
def evaluate_model(true, predicted):
    acc = accuracy_score(true, predicted)
    precision = precision_score(true, predicted)
    recall = recall_score(true, predicted)
    f1 = f1_score(true, predicted)
    
    return acc, precision, recall, f1

In [27]:
models = {
    "Logistic Regression": LogisticRegression(),
    "KNN": KNeighborsClassifier(),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
    "AdaBoost": AdaBoostClassifier(),
    "SVM": SVC(probability=True),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss'),
    "CatBoost": CatBoostClassifier(verbose=False)
}

results = []

for name, model in models.items():
    # Train
    model.fit(X_train, y_train)
    
    # Predict
    y_pred = model.predict(X_test)
    
    # Probabilities (for ROC-AUC)
    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)[:, 1]
        roc_auc = roc_auc_score(y_test, y_prob)
    else:
        roc_auc = None
    
    # Metrics
    acc = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    results.append({
        "Model": name,
        "Accuracy": acc,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "ROC-AUC": roc_auc
    })

# Convert to DataFrame
results_df = pd.DataFrame(results)

# Sort by best model
results_df = results_df.sort_values(by="F1 Score", ascending=False)

print(results_df)

                 Model  Accuracy  Precision    Recall  F1 Score   ROC-AUC
0  Logistic Regression  0.790334   0.627832  0.518717  0.568082  0.832358
7             CatBoost  0.795309   0.653571  0.489305  0.559633  0.834799
4             AdaBoost  0.778962   0.602606  0.494652  0.543319  0.826914
5                  SVM  0.791756   0.651685  0.465241  0.542902  0.793816
3        Random Forest  0.787491   0.637363  0.465241  0.537867  0.818140
6              XGBoost  0.769012   0.575851  0.497326  0.533716  0.803216
2        Decision Tree  0.759773   0.552023  0.510695  0.530556  0.680323
1                  KNN  0.746269   0.522427  0.529412  0.525896  0.762536
